[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zshakespear/bd-spectra-onramp/blob/main/notebooks/05_global_parameters.ipynb)

# 5 · Global atmospheric parameters

*Brown Dwarf Spectra On-Ramp · about 50 minutes*

**By the end of this notebook you will be able to:**
- say what effective temperature, gravity, metallicity, clouds, and vertical mixing each do to a spectrum
- read a parameter stack and a ratio-to-baseline plot
- compare real low-gravity and low-metallicity objects with field objects of the same type
- recognize degeneracies: different parameter changes that look alike

> **First time here?** Do *File → Save a copy in Drive* before you change anything (notebook 0 explains why).
> Then run the **Setup** cell below. Run cells in order, top to bottom, with **Shift + Enter**.
> **Moving between notebooks:** use the links inside the notebooks (such as **Next** at the bottom). They always open
> the original notebook, so they work from your copy too — then save a copy of the new notebook before you edit it.

In [ ]:
# ▶ SETUP - run this cell first. It is safe to run more than once.
REPO_URL = "https://github.com/zshakespear/bd-spectra-onramp.git"
import os, subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules:                       # running in Google Colab
    ROOT = Path("/content/bd-spectra-onramp")
    if not (ROOT / "bdtools").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only", "--quiet"], check=False)
else:                                                   # running in Jupyter on your own computer
    ROOT = Path.cwd()
    while not (ROOT / "bdtools").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import bdtools as bd
bd.setup_notebook()

## 1. Model grids

A **model grid** is a large set of computed spectra, each for one combination of a few **global parameters**:

| Parameter | Symbol | What it is |
|---|---|---|
| Effective temperature | T<sub>eff</sub> | the temperature of a blackbody with the same total emitted power |
| Surface gravity | log g | log<sub>10</sub> of g in cm s<sup>−2</sup>; young, low-mass objects have low gravity |
| Metallicity | [M/H] | log<sub>10</sub> of the heavy-element fraction relative to the Sun (0 = solar, −0.5 ≈ ⅓ solar) |
| Clouds | f<sub>sed</sub> | how quickly cloud particles **rain out**: large f<sub>sed</sub> = fast, so thin clouds; small f<sub>sed</sub> = slow, so thick, extended clouds; "nc" = cloud-free (section 5) |
| Vertical mixing | K<sub>zz</sub> | how fast gas is stirred up and down inside the atmosphere (section 6) |

We use a curated subset of model grids: **Sonora Diamondback** (cloudy, equilibrium chemistry; L and T),
**Sonora Flame Skimmer** (cloud-free, disequilibrium chemistry with a K<sub>zz</sub> axis; late T and Y) and **Exo-REM**
(clouds and disequilibrium chemistry). Only Flame Skimmer's 500–550 K models have been downloaded so far, so its
temperature sweep is narrow.
The models were averaged to modest resolution (R ~ 500) once, by your instructor. **Never compare a high-resolution
model directly with SPHEREx data** — bdtools always bins it onto the data's own channels first (notebook 6).

The library is organised around three **baseline** models — a typical mid/late-L dwarf, an early-T dwarf and a
late-T/Y dwarf. Each **sweep** starts from a baseline and changes only *one* parameter, keeping the rest fixed.

In [ ]:
lib = bd.optional(bd.list_models)
lib if lib is None else lib[lib["sweep"] == "baseline"]

Each plot below has two panels: the spectra (thick line = baseline) and each model **divided by the baseline**,
with the absorption bands shaded. The ratio panel is where you read the effect of a parameter.

## 2. Effective temperature

**Predict:** a cooler model at the same gravity — will it be fainter everywhere by the same factor?

In [ ]:
r = bd.optional(bd.load_sweep, "L", "teff")
if r is not None:
    baseline, members = r
    bd.plot_parameter_stack(baseline, members, "teff", value_label="Teff [K]", log=True,
                            title="mid/late-L baseline model: only Teff [K] changes")

In [ ]:
r = bd.optional(bd.load_sweep, "T_early", "teff")
if r is not None:
    baseline, members = r
    bd.plot_parameter_stack(baseline, members, "teff", value_label="Teff [K]", log=True,
                            title="early-T baseline model: only Teff [K] changes")

Temperature is a **global** parameter: it changes the overall slope (Planck, notebook 1) *and* the depths of
many bands at once, because the chemistry and the temperature profile both change.

**Careful: "only T<sub>eff</sub> changes" describes the model's inputs, not its physics.** A grid lets you lower T<sub>eff</sub>
while holding the other *numbers* fixed, but the atmosphere itself still changes in ways that are not held fixed.
For example, as an L dwarf cools, its silicate clouds form deeper and eventually sink below the layers we see (they
**rain out** of view), and carbon shifts from CO toward CH<sub>4</sub>. So one step in T<sub>eff</sub> can carry a change in clouds
and chemistry along with it.

## 3. Surface gravity

In [ ]:
r = bd.optional(bd.load_sweep, "L", "logg")
if r is not None:
    baseline, members = r
    bd.plot_parameter_stack(baseline, members, "logg", value_label="log g", log=True,
                            title="mid/late-L baseline model: only log g changes")

In [ ]:
r = bd.optional(bd.load_sweep, "T_early", "logg")
if r is not None:
    baseline, members = r
    bd.plot_parameter_stack(baseline, members, "logg", value_label="log g", log=True,
                            title="early-T baseline model: only log g changes")

Higher gravity means higher pressure at the photosphere, which strengthens pressure-dependent absorption such as
H<sub>2</sub> CIA (≈1.7–3 µm; notebook 3). Watch the K-band region. The observed signs of **low gravity** (Rustamkulov et al.
2026): a shallower 0.75–1.2 µm slope, weaker water features, and a taller K peak relative to H.

**Gravity is a tricky parameter to fit.** Its effects are modest, and other parameters can produce similar ones. When
two different parameter changes produce nearly the **same** change in the spectrum, they are **degenerate**: the data
cannot tell them apart, so a fit can trade one for the other — for example, a slightly different temperature,
metallicity or cloud thickness can partly mimic a change in gravity. Section 9 shows an example. Expect a fitted
log g to be uncertain.

## 4. Metallicity

In [ ]:
r = bd.optional(bd.load_sweep, "L", "mh")
if r is not None:
    baseline, members = r
    bd.plot_parameter_stack(baseline, members, "mh", value_label="[M/H]", log=True,
                            title="mid/late-L baseline model: only [M/H] changes")

In [ ]:
r = bd.optional(bd.load_sweep, "T_early", "mh")
if r is not None:
    baseline, members = r
    bd.plot_parameter_stack(baseline, members, "mh", value_label="[M/H]", log=True,
                            title="early-T baseline model: only [M/H] changes")

Fewer heavy elements → less H<sub>2</sub>O, CH<sub>4</sub>, CO, … but H<sub>2</sub> (and its CIA) is untouched. In observed
**low-metallicity** objects the J (1.3 µm) to K (2.2 µm) peak ratio reverses, because H<sub>2</sub> CIA dominates from
~1.7 to 3 µm (Rustamkulov et al. 2026).

## 5. Clouds

In [ ]:
r = bd.optional(bd.load_sweep, "L", "clouds")
if r is not None:
    baseline, members = r
    bd.plot_parameter_stack(baseline, members, "clouds", value_label="f_sed", log=True,
                            title="mid/late-L baseline model: only f_sed changes")

In [ ]:
r = bd.optional(bd.load_sweep, "T_early", "clouds")
if r is not None:
    baseline, members = r
    bd.plot_parameter_stack(baseline, members, "clouds", value_label="f_sed", log=True,
                            title="early-T baseline model: only f_sed changes")

Thick clouds (small f<sub>sed</sub>) block the deep windows, so the window peaks drop and the spectrum looks redder and
smoother; cloud-free models have the deepest, brightest windows.

**The L/T transition.** Across the L/T transition the J band (1.27 µm) *brightens* even though the objects are
cooling. The leading explanation is **cloud clearing**: the clouds thin out or break up, letting light from deeper,
hotter layers escape, but there is a lot of uncertainty here. The model grids handle the transition in different ways.

**What f<sub>sed</sub> means.** Cloud particles are always being lifted by the moving gas and are always settling back
down under gravity. f<sub>sed</sub> (the *sedimentation efficiency*) says how quickly they settle — how fast the clouds
**rain out**:

* **large f<sub>sed</sub>:** particles grow big and rain out quickly, so the cloud is thin and stays in a narrow layer near
  where it forms;
* **small f<sub>sed</sub>:** particles stay small and are carried high before they fall, so the cloud is thick and extends
  far up — where it blocks more of the light from below.

## 6. Vertical mixing (K<sub>zz</sub>)

The atmosphere is not still: convection and turbulence keep carrying gas up and down. This is **vertical mixing**,
and K<sub>zz</sub> measures how fast it happens (we quote log<sub>10</sub> K<sub>zz</sub>, with K<sub>zz</sub> in cm² s⁻¹; bigger = faster).

Why it matters: which molecules are present depends on temperature. Deep down it is hot, and carbon is mostly in
**CO**; high up it is cold, and carbon "wants" to be in **CH<sub>4</sub>**. Converting CO into CH<sub>4</sub> takes time. If mixing
lifts gas faster than that conversion can happen, CO survives up into the cold layers we see. The atmosphere then
has more CO (and less CH<sub>4</sub>) than equilibrium chemistry predicts — this is **disequilibrium chemistry**. Stronger
mixing means more of it.

In [ ]:
r = bd.optional(bd.load_sweep, "T_late_Y", "kzz")
if r is not None:
    baseline, members = r
    bd.plot_parameter_stack(baseline, members, "kzz", value_label="log Kzz", log=True,
                            title="late-T/Y baseline model: only log Kzz changes")

Stronger mixing dredges up CO (and CO<sub>2</sub>) from hot deep layers faster than chemistry can convert it to CH<sub>4</sub>,
so the 4.3–4.9 µm notches deepen and the 3.3 µm CH<sub>4</sub> band changes. The Diamondback grid assumes equilibrium
chemistry and has no K<sub>zz</sub> axis.

The late-T/Y baseline's other sweeps:

In [ ]:
r = bd.optional(bd.load_sweep, "T_late_Y", "teff")
if r is not None:
    baseline, members = r
    bd.plot_parameter_stack(baseline, members, "teff", value_label="Teff [K]", log=True,
                            title="late-T/Y baseline model: only Teff [K] changes")

In [ ]:
r = bd.optional(bd.load_sweep, "T_late_Y", "logg")
if r is not None:
    baseline, members = r
    bd.plot_parameter_stack(baseline, members, "logg", value_label="log g", log=True,
                            title="late-T/Y baseline model: only log g changes")

## 7. Grids disagree

Even with the **same** parameters, different grids differ by up to about a factor of 2 at some wavelengths
(Rustamkulov et al. 2026). And even the best-matching model rarely reproduces a real spectrum within its
uncertainties (notebook 6). **Treat parameters read off a model grid as rough.**

"Same parameters" means the same *numbers*, not the same physics: each grid has its own cloud recipe (so
f<sub>sed</sub> = 4 in one grid is not the same cloud as f<sub>sed</sub> = 4 in another), its own chemistry
(Diamondback assumes equilibrium; Exo-REM includes disequilibrium chemistry), and its own opacities. The Exo-REM
spectra in this library also start near 0.93 µm, so the shortest wavelengths show only one grid.

In [ ]:
lib = bd.optional(bd.list_models)
if lib is not None:
    for base in lib["baseline"].unique():
        cg = lib[(lib["baseline"] == base) & (lib["sweep"] == "cross_grid")]
        b0 = lib[(lib["baseline"] == base) & (lib["sweep"] == "baseline")]
        if cg.empty or b0.empty:
            continue
        a = bd.at_spherex_resolution(bd.load_model(b0.iloc[0]["id"]))
        b = bd.at_spherex_resolution(bd.load_model(cg.iloc[0]["id"]))
        fig, ax = bd.plot_spectrum([a, b], log=True, title=f"{base}: two grids, same nominal parameters (SPHEREx resolution)")
        print(base, "largest ratio between the grids:", round(float(np.nanmax(a.flux / b.flux)), 2),
              " smallest:", round(float(np.nanmin(a.flux / b.flux)), 2))

## 8. Real low-gravity and low-metallicity objects

Rustamkulov et al. (2026) include a **low-gravity** sequence at L0 and L5 (their Table 2: field, β = intermediate
gravity, γ = low gravity, i.e. young) and a **low-metallicity** sequence at L0 and L7 (Table 3: field, sd = subdwarf,
esd = extreme subdwarf). Each group below has the same spectral type, so the differences come from gravity or
metallicity rather than temperature. All are normalized at the J peak.

**Predict** before you run it, using sections 3 and 4:
* low gravity: what happens to the 0.75–1.2 µm slope, the water bands, and the K peak relative to H?
* low metallicity: what happens to the K peak (≈2.2 µm) relative to J (≈1.3 µm)?

> Three objects in Tables 2 and 3 are not in the public SPIFF release. The field L0 in both groups is a **stand-in**
> (the Table 1 L0, which the paper itself preferred over the L0 standard for its S/N), and so is the esdL0 (an
> esdL0.5, with low S/N). There is no usdL0 (no ultra-subdwarfs at all in SPIFF). `bd.list_references()` marks
> stand-ins with `status` = `substitute`.

In [ ]:
refs = bd.optional(bd.list_references)
if refs is not None:
    for seqname in ["low_gravity", "low_metallicity"]:
        rows = refs[refs["sequence"] == seqname]
        for base in rows["base_type"].unique():
            group = rows[rows["base_type"] == base].sort_values("order")
            specs = [bd.load_reference(i) for i in group["id"]]
            bd.plot_spectrum(specs, log=True, normalize_at=(1.2, 1.3), errors=False,
                             labels=[s.name.split(" ")[0] for s in specs],
                             title=f"{base}: {seqname.replace('_', ' ')} sequence (normalized at 1.2-1.3 um)")

## 9. Degeneracies: different causes, similar effects

Two parameters are **degenerate** when changing either one changes the spectrum in nearly the same way (section 3).
Here we take a baseline model and change it in two different ways — make it **cooler**, or give it **higher
gravity** — and ask: could you tell which one happened?

Both changed models are scaled to match the baseline at the J peak, so only their *shapes* differ. In the bottom
panel, each changed model is divided by the baseline: 1 = same shape as the baseline.

In [ ]:
# ✏️ SETTINGS YOU CAN CHANGE
BASELINE = "L"            # "L" (mid/late-L) or "T_early" (early-T): try both

In [ ]:
lib = bd.optional(bd.list_models, baseline=BASELINE)
if lib is not None and not lib.empty:
    base = bd.load_model(lib[lib["sweep"] == "baseline"].iloc[0]["id"])
    cool = lib[(lib["sweep"] == "teff") & (lib["teff"] < base.meta["teff"])].sort_values("teff").tail(1)
    heavy = lib[(lib["sweep"] == "logg") & (lib["logg"] > base.meta["logg"])].sort_values("logg").head(1)
    if not cool.empty and not heavy.empty:
        b = bd.at_spherex_resolution(base)
        c = bd.at_spherex_resolution(bd.load_model(cool.iloc[0]["id"]))
        h = bd.at_spherex_resolution(bd.load_model(heavy.iloc[0]["id"]))
        at_j = lambda s: s.flux / np.nanmean(s.flux[(s.wave_um >= 1.2) & (s.wave_um <= 1.3)])
        lab_b = f"baseline: T_eff = {b.meta['teff']:g} K, log g = {b.meta['logg']:g}"
        lab_c = f"cooler: T_eff = {c.meta['teff']:g} K"
        lab_h = f"higher gravity: log g = {h.meta['logg']:g}"
        fig, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 8), gridspec_kw={"height_ratios": [2, 1.6]})
        for ax in axes:
            bd.shade_bands(ax)
        bd.add_band_key(axes[0])
        axes[0].plot(b.wave_um, at_j(b), color="k", lw=2.2, label=lab_b)
        axes[0].plot(c.wave_um, at_j(c), color=bd.LINE_COLORS[1], lw=1.4, label=lab_c)
        axes[0].plot(h.wave_um, at_j(h), color=bd.LINE_COLORS[2], lw=1.4, label=lab_h)
        axes[0].set_yscale("log"); axes[0].set_ylabel("flux / flux at the J peak"); axes[0].legend(fontsize=8)
        axes[1].plot(c.wave_um, at_j(c) / at_j(b), "o-", ms=3, color=bd.LINE_COLORS[1], label=lab_c)
        axes[1].plot(h.wave_um, at_j(h) / at_j(b), "o-", ms=3, color=bd.LINE_COLORS[2], label=lab_h)
        axes[1].axhline(1, color="0.4")
        axes[1].set_ylabel("changed / baseline\n(both = 1 at the J peak)"); axes[1].legend(fontsize=8)
        axes[1].set_xlabel("Wavelength [µm]"); axes[1].set_xlim(0.7, 5.1)
        axes[0].set_title("Cooler, or higher gravity? Both scaled to match the baseline at the J peak", fontsize=11, pad=24)

Read the bottom panel. **Where the two curves move together**, a cooler object and a higher-gravity object look the
same: those wavelengths cannot tell the two apart (degenerate). **Where the curves split**, the data *can* tell them
apart — those are the wavelengths that break the degeneracy. Which bands are they, and which absorbers sit there?

For the mid/late-L baseline, both changes make the spectrum redder relative to J over most of the range — a partial
degeneracy. Now set `BASELINE = "T_early"` and re-run: for the early-T model the two changes look quite different.
**Whether two parameters are degenerate depends on the kind of object**, which is one reason fitted parameters are
more trustworthy for some objects than for others.

Unresolved **binarity** (two objects blended into one spectrum) can also mimic a parameter change — see notebook 6.

### Exercise 5.1
In the T<sub>eff</sub> sweep, which wavelength range changes most in ratio? Why does that make sense from notebook 1?

<details><summary><b>Answer</b> (try first, then click)</summary>

The short wavelengths (and the deep windows) change most, because they are on the steep, short-wavelength side of
the Planck function, where flux is most sensitive to temperature.

</details>

### Exercise 5.2
Name two things that can make the K peak taller relative to H.

<details><summary><b>Answer</b> (try first, then click)</summary>

Lower gravity (less H<sub>2</sub> CIA; Rustamkulov et al. 2026 list a taller K peak relative to H as a low-gravity sign) and
thicker clouds / redder colours. Metallicity matters too. That's a degeneracy — use other bands to break it.

</details>

## Summary

- T<sub>eff</sub>: global — slope and many bands. log g and [M/H]: act strongly through H<sub>2</sub> CIA (~1.7–3 µm) and band depths.
- Clouds (f<sub>sed</sub>) dim the windows and redden; K<sub>zz</sub> sets how much CO/CO<sub>2</sub> survives in cold atmospheres.
- Grids with the same parameters can differ by ~2×, and no model matches a real spectrum perfectly: treat parameters as rough.
- Holding the other numbers fixed does not hold the physics fixed (e.g. clouds rain out of view as T<sub>eff</sub> drops).
- Degeneracy: two parameter changes that look alike. Gravity is especially hard to pin down; look for bands that respond differently.
- Field guide: [Global parameters](https://github.com/zshakespear/bd-spectra-onramp/blob/main/field_guide/parameters/README.md)

**Field guide:** the pages linked above are the short versions to keep open while you work.

## Self-check

Answer these without scrolling up. If one is hard, revisit that section.

1. Which parameter changes the spectrum most globally?
<details><summary><b>Answer</b> (try first, then click)</summary>

Effective temperature.

</details>

2. Name one observed sign of low gravity.
<details><summary><b>Answer</b> (try first, then click)</summary>

A shallower 0.75–1.2 µm slope, weaker water features, or a taller K peak relative to H.

</details>

3. What does vertical mixing do to CO in a cold atmosphere?
<details><summary><b>Answer</b> (try first, then click)</summary>

Brings it up from hot deep layers faster than it converts to CH<sub>4</sub>, so more CO absorption.

</details>

4. Why does a small f<sub>sed</sub> mean thick clouds?
<details><summary><b>Answer</b> (try first, then click)</summary>

Particles rain out slowly, so they stay aloft and spread high into the atmosphere.

</details>

5. Two grids give different spectra for the same parameters. What should you conclude?
<details><summary><b>Answer</b> (try first, then click)</summary>

Model physics differs; treat best-fit parameters as rough.

</details>

**Next:** [Notebook 6 · Fitting models to spectra](https://colab.research.google.com/github/zshakespear/bd-spectra-onramp/blob/main/notebooks/06_fitting_models.ipynb)